In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np

df = pd.read_csv("../../data/raw/2020-Apr-L.csv")

In [2]:
df['event_time'] = pd.to_datetime(df['event_time'], utc=True)

df["day_of_week"] = df["event_time"].dt.day_name()
 
orden = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
labels_es = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

dia_counts = df.groupby("day_of_week").size().reindex(orden)
 
days   = list(range(7))
values = dia_counts.values
 
max_dia  = dia_counts.idxmax()
min_dia  = dia_counts.idxmin()
max_val  = dia_counts[max_dia]
min_val  = dia_counts[min_dia]
mean_val = dia_counts.mean()
 
max_idx = orden.index(max_dia)
min_idx = orden.index(min_dia)

In [3]:
# PALETA: laborables vs fines de semana
# degradado por intensidad dentro de cada grupo
es_finde = [False, False, False, False, False, True, True]
 
norm_lab  = plt.Normalize(vmin=min(values[:5]), vmax=max(values[:5]))
norm_fin  = plt.Normalize(vmin=min(values[5:]), vmax=max(values[5:]))
cmap_lab  = plt.cm.Blues
cmap_fin  = plt.cm.Purples
 
colors = []
for i, (val, finde) in enumerate(zip(values, es_finde)):
    if i == max_idx:
        colors.append("#1D4ED8")       # azul oscuro — máximo global
    elif i == min_idx:
        colors.append("#FCA5A5")       # rojo suave  — mínimo global
    elif finde:
        colors.append(cmap_fin(0.45 + 0.4 * norm_fin(val)))
    else:
        colors.append(cmap_lab(0.35 + 0.45 * norm_lab(val)))

In [4]:
# FIGURA
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor("#F8FAFC")
ax.set_facecolor("#F8FAFC")
 
bars = ax.bar(days, values, color=colors, width=0.6,
              edgecolor="white", linewidth=0.8, zorder=3)
 
# Fondo sombreado fines de semana
ax.axvspan(4.65, 6.35, color="#EDE9FE", alpha=0.4, zorder=1, label="_nolegend_")
ax.text(5.5, max_val * 1.01, "Fin de semana",
        ha="center", fontsize=7.5, color="#7C3AED", style="italic")
 
# Línea de media
ax.axhline(mean_val, color="#64748B", linewidth=1.2,
           linestyle="--", zorder=2)
ax.text(6.55, mean_val * 1.01, f"Media\n{int(mean_val):,}",
        fontsize=7.5, color="#64748B", va="bottom")
 
# Etiquetas en todas las barras (diagonal)
for i, (bar, val) in enumerate(zip(bars, values)):
    es_max = (i == max_idx)
    es_min = (i == min_idx)
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_val * 0.008,
        f"{int(val):,}",
        ha="left", va="bottom",
        fontsize=7.5 if (es_max or es_min) else 7,
        color="#1D4ED8" if es_max else ("#DC2626" if es_min else "#334155"),
        fontweight="bold" if (es_max or es_min) else "normal",
        rotation=45
    )
 
# Ejes
ax.set_xticks(days)
ax.set_xticklabels(labels_es, fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"{int(x):,}"))
ax.tick_params(axis="both", length=0)
ax.set_ylabel("Número de Interacciones", fontsize=10, color="#334155", labelpad=10)
ax.set_xlabel("Día de la Semana", fontsize=10, color="#334155", labelpad=10)
 
# Cuadrícula horizontal suave
for y in ax.get_yticks():
    ax.axhline(y, color="#E2E8F0", linewidth=0.7, zorder=1)
 
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#E2E8F0")
ax.spines["bottom"].set_color("#E2E8F0")
 
ax.set_xlim(-0.5, 6.8)
 
# Títulos
ax.set_title("Ciclos Semanales — Interacciones por Día de la Semana",
             fontsize=13, fontweight="bold", loc="left",
             pad=28, color="#0F172A")
ax.text(0.0, 1.045,
        "Volumen agregado de eventos · Datos: 2020-04-01 → 2020-04-30 (UTC)",
        transform=ax.transAxes, fontsize=8.5, color="#64748B")
 
# Leyenda grupos
patch_lab = mpatches.Patch(color=cmap_lab(0.6), label="Días laborables")
patch_fin = mpatches.Patch(color=cmap_fin(0.6), label="Fin de semana")
ax.legend(handles=[patch_lab, patch_fin],
          fontsize=8.5, frameon=False,
          loc="upper left", bbox_to_anchor=(0.01, 0.95))
 
# Estadísticas resumen
stats_text = (
    f"n total = {int(dia_counts.sum()):,}\n"
    f"Media/día = {int(mean_val):,}"
)
ax.text(0.99, 0.97, stats_text,
        transform=ax.transAxes,
        fontsize=8, color="#475569",
        va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.4", fc="#F1F5F9", ec="#CBD5E1", lw=1))
 
plt.tight_layout()
plt.savefig("outputs/04_temporal_weekly_patterns.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print("¡Gráfico 04_temporal_weekly_patterns.png guardado exitosamente!")

¡Gráfico 04_temporal_weekly_patterns.png guardado exitosamente!
